In [306]:

from typing import TypedDict, Literal, Annotated

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
import operator
from pprint import pprint

load_dotenv()

True

In [307]:
class TweetState(TypedDict):
    topic: str
    tweet: Annotated[list[str], operator.add]

    evaluation: Literal["approved", "needs_improvement"]
    feedback: Annotated[list[str], operator.add]

    iteration: int
    max_iterations: int


In [308]:
llm = ChatGroq(model="qwen/qwen3-32b", temperature=0.4)

In [309]:
class TweetOutputParser(BaseModel):
    tweet: str = Field(description="Tweet to be tweeted.")

In [310]:
class EvaluateOutputParser(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(description="Evaluation of the tweet.")
    feedback: str = Field(description="Feedback on the tweet.")

In [311]:
tweet_structured_llm = llm.with_structured_output(TweetOutputParser)
evaluate_structured_llm = llm.with_structured_output(EvaluateOutputParser)

In [312]:
def generate_tweet(state: TweetState) -> TweetState:
    response = tweet_structured_llm.invoke(
        input=f"Generate a tweet of about 250 words about the following topic: {state['topic']}.")
    return {"tweet": [response.tweet]}


In [313]:
def evaluate_tweet(state: TweetState) -> TweetState:
    response = evaluate_structured_llm.invoke(
        input=f"Evaluate the following tweet of the basis of creativity, uniqueness, and engagement: {state['tweet'][-1]}")
    return {"evaluation": response.evaluation, "feedback": [response.feedback]}

In [314]:
def optimize_tweet(state: TweetState) -> TweetState:
    response = tweet_structured_llm.invoke(
        input=f"Optimize the following tweet: {state['tweet'][-1]} on the basis of the following feedback : {state['feedback'][-1]}. Shouldn't exceed 250 words.")
    return {"tweet": [response.tweet], "iteration": state["iteration"]+1}

In [315]:
def get_conditional_node(state: TweetState):
    if state["evaluation"] == "needs_improvement" and state["iteration"] <= state["max_iterations"]:
        return "needs_improvement"
    return "approved"

In [316]:
graph = StateGraph(TweetState)

graph.add_node("generate_tweet", generate_tweet)
graph.add_node("evaluate_tweet", evaluate_tweet)
graph.add_node("optimize_tweet", optimize_tweet)

graph.add_edge(START, "generate_tweet")
graph.add_edge("generate_tweet", "evaluate_tweet")

graph.add_conditional_edges("evaluate_tweet", get_conditional_node, {"needs_improvement": "optimize_tweet", "approved": END})
graph.add_edge("optimize_tweet", "evaluate_tweet")


workflow = graph.compile()



In [317]:
initial_state = {
    "topic": "I have started to learn LangGraph and this post is generated by my first agent It should be really exiciting",
    "iteration": 0,
    "max_iterations": 5
}

final_state = workflow.invoke(initial_state)

final_state

{'topic': 'I have started to learn LangGraph and this post is generated by my first agent It should be really exiciting',
 'tweet': ["Exciting news! I've started learning LangGraph, and this tweet is generated by my first agent! 🚀 LangGraph is an awesome framework for building AI agents, and I'm thrilled to explore its potential. My agent can already handle tasks and answer questions. The future of AI is bright, and I can't wait to see where this journey takes me! #LangGraph #AI #MachineLearning #TechInnovation",
  'Learning LangGraph feels like teaching a robot to dance—chaotic at first, but with the right code, it can glide! 🤖✨ My first agent just drafted this tweet, so I’m both impressed and slightly terrified. (Who knew AI could be a drama queen?) It can already handle tasks and answer questions—imagine a robot barista that remembers your order AND roasts your coding skills. ☕💻 What’s *your* first AI project? Let’s geek out! #LangGraph #AI #MachineLearning #TechInnovation'],
 'eval